In [ ]:
# ══════════════════════════════════════════════════════════════════
# Colab setup — self-contained. No git clone, no FORK_URL, no sys.path.
# Before running: Runtime → Change runtime type → T4 GPU (free tier)
# ══════════════════════════════════════════════════════════════════
!pip install -q kailash polars plotly gdown python-dotenv nest-asyncio kailash-ml

import nest_asyncio; nest_asyncio.apply()

import torch
if torch.cuda.is_available():
    print(f'✓ GPU available: {torch.cuda.get_device_name(0)}')
else:
    print('⚠ No GPU — training will be slow. Runtime → Change runtime type → T4 GPU')

print('✓ Setup complete. All helpers defined in the next cell.')


In [ ]:
# ══════════════════════════════════════════════════════════════════
# MLFP inlined helpers — DO NOT EDIT (collapse this cell!)
# Auto-generated by scripts/generate_selfcontained_notebook.py for mlfp04
# ══════════════════════════════════════════════════════════════════
from __future__ import annotations

# ── shared/kailash_helpers.py ──
"""Common Kailash SDK setup patterns for MLFP exercises."""


import os
from pathlib import Path

from dotenv import load_dotenv


def setup_environment() -> None:
    """Load .env and validate common configuration.

    Call this at the top of every exercise that needs API keys or DB connections.
    """
    # Find .env by walking up from the exercise file
    env_path = Path.cwd() / ".env"
    if not env_path.exists():
        # Try repo root
        for parent in Path.cwd().parents:
            candidate = parent / ".env"
            if candidate.exists():
                env_path = candidate
                break

    load_dotenv(env_path)


def get_connection_manager(db_url: str | None = None):
    """Create a ConnectionManager for kailash-ml engines.

    Args:
        db_url: Database URL. Defaults to SQLite at ./mlfp.db
    """
    from kailash.db import ConnectionManager

    url = db_url or os.environ.get("DATABASE_URL", "sqlite:///mlfp.db")
    return ConnectionManager(url)


def get_device() -> "torch.device":
    """Get the best available compute device: MPS (Mac) > CUDA > CPU."""
    import torch

    if hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
        return torch.device("mps")
    if torch.cuda.is_available():
        return torch.device("cuda")
    return torch.device("cpu")


def get_llm_model() -> str:
    """Get the configured LLM model name from environment."""
    setup_environment()
    model = os.environ.get("DEFAULT_LLM_MODEL", os.environ.get("OPENAI_PROD_MODEL"))
    if not model:
        raise EnvironmentError(
            "No LLM model configured. Set DEFAULT_LLM_MODEL or OPENAI_PROD_MODEL in .env"
        )
    return model


# ── shared/data_loader.py ──
"""Unified data loading for MLFP course — supports local and Colab."""


import logging
from pathlib import Path

import polars as pl

logger = logging.getLogger(__name__)

# Google Drive shared folder containing all MLFP datasets
_DRIVE_FOLDER_ID = "16c3RkGmiwMWbjD7cJKbJx-JRZlgmQdws"

# Module subfolders on the shared Drive
_MODULES = {
    "mlfp01",
    "mlfp02",
    "mlfp03",
    "mlfp04",
    "mlfp05",
    "mlfp06",
    "mlfp_assessment",
}


def _is_colab() -> bool:
    """Detect if running inside Google Colab."""
    try:
        import google.colab  # noqa: F401

        return True
    except ImportError:
        return False


def _colab_data_root() -> Path:
    """Return the Drive-mounted mlfp_data path in Colab."""
    return Path("/content/drive/MyDrive/mlfp_data")


def _local_cache_dir() -> Path:
    """Return local cache directory for downloaded files."""
    cache = Path.cwd() / ".data_cache"
    cache.mkdir(exist_ok=True)
    return cache


def _download_from_drive(module: str, filename: str, dest: Path) -> Path:
    """Download a file from the shared Google Drive using gdown."""
    import gdown

    dest_dir = dest / module
    dest_dir.mkdir(parents=True, exist_ok=True)
    dest_file = dest_dir / filename

    if dest_file.exists():
        logger.debug("Using cached file: %s", dest_file)
        return dest_file

    # gdown can download from a folder by file path
    url = f"https://drive.google.com/drive/folders/{_DRIVE_FOLDER_ID}"
    logger.info("Downloading %s/%s from Google Drive...", module, filename)

    # Download the specific file from the shared folder
    try:
        gdown.download_folder(
            url=url,
            output=str(dest),
            quiet=True,
            remaining_ok=True,
        )
    except TypeError:
        # Older gdown versions don't support remaining_ok
        gdown.download_folder(
            url=url,
            output=str(dest),
            quiet=True,
        )

    if not dest_file.exists():
        # Try direct download if folder download didn't isolate the file
        for candidate in dest.rglob(filename):
            if candidate.is_file():
                if candidate != dest_file:
                    candidate.rename(dest_file)
                return dest_file

        msg = (
            f"File not found after download: {module}/{filename}. "
            f"Check that it exists in the mlfp_data shared Drive."
        )
        raise FileNotFoundError(msg)

    return dest_file


def _read_file(path: Path) -> pl.DataFrame:
    """Read a data file into a polars DataFrame."""
    suffix = path.suffix.lower()
    if suffix == ".csv":
        return pl.read_csv(path, try_parse_dates=True)
    elif suffix == ".parquet":
        return pl.read_parquet(path)
    elif suffix == ".json":
        return pl.read_json(path)
    elif suffix in (".p", ".pickle", ".pkl"):
        import pickle

        with open(path, "rb") as f:
            obj = pickle.load(f)  # noqa: S301
        if isinstance(obj, pl.DataFrame):
            return obj
        raise TypeError(
            f"Cannot convert pickle object of type {type(obj)} to polars DataFrame. "
            f"Convert the pickle to parquet upstream: pl.from_pandas(obj).write_parquet('out.parquet')"
        )
    else:
        raise ValueError(
            f"Unsupported file format: {suffix}. Use .csv, .parquet, or .json"
        )


def _repo_data_dir() -> Path | None:
    """Find the repo-local data/ directory by walking up from cwd."""
    for parent in [Path.cwd(), *Path.cwd().parents]:
        candidate = parent / "data"
        if candidate.is_dir() and (parent / "pyproject.toml").exists():
            return candidate
    return None


class MLFPDataLoader:
    """Load MLFP course datasets with automatic source resolution.

    Resolution order:
    1. Colab: Drive mount at /content/drive/MyDrive/mlfp_data/
    2. Local repo data/ directory (committed datasets)
    3. Google Drive download via gdown (cached in .data_cache/)

    Usage:
        loader = MLFPDataLoader()
        df = loader.load("mlfp01", "hdbprices.csv")

    Shortcut:
        df = MLFPDataLoader.mlfp01("hdbprices.csv")
    """

    def __init__(self, cache_dir: Path | str | None = None):
        self._colab = _is_colab()
        if self._colab:
            self._root = _colab_data_root()
        else:
            self._local_data = _repo_data_dir()
            self._cache = Path(cache_dir) if cache_dir else _local_cache_dir()

    def load_raw(self, module: str, filename: str) -> Path:
        """Return the file path without reading into memory.

        Use this for image directories, audio files, or any data that torch/HF
        loads directly rather than via polars.

        Args:
            module: Module subfolder (e.g., "mlfp05")
            filename: File or directory name (e.g., "fashion_mnist", "cifar10")

        Returns:
            Path to the local file or directory.
        """
        if module not in _MODULES:
            raise ValueError(
                f"Unknown module '{module}'. Available: {sorted(_MODULES)}"
            )

        if self._colab:
            path = self._root / module / filename
        else:
            if self._local_data:
                local_path = self._local_data / module / filename
                if local_path.exists():
                    return local_path
            path = self._cache / module / filename

        if not path.exists():
            raise FileNotFoundError(
                f"Raw data not found: {module}/{filename}. "
                f"Run 'python scripts/fetch-real-data.py' to download."
            )
        return path

    @staticmethod
    def load_hf(
        dataset_name: str,
        split: str = "train",
        streaming: bool = False,
    ):
        """Load a HuggingFace dataset directly (not via polars).

        Use this for large datasets (millions of rows) or multimodal data
        (images, audio) that don't fit into a DataFrame.

        Args:
            dataset_name: HuggingFace dataset ID (e.g., "zalando-datasets/fashion_mnist")
            split: Dataset split ("train", "test", "validation")
            streaming: If True, returns an IterableDataset for memory-efficient processing

        Returns:
            HuggingFace Dataset or IterableDataset object.
        """
        from datasets import load_dataset

        logger.info(
            "Loading HuggingFace dataset: %s (split=%s, streaming=%s)",
            dataset_name,
            split,
            streaming,
        )
        return load_dataset(dataset_name, split=split, streaming=streaming)

    def load(self, module: str, filename: str) -> pl.DataFrame:
        """Load a dataset file as a polars DataFrame.

        Args:
            module: Module subfolder (e.g., "mlfp01", "mlfp_assessment")
            filename: File name within the module folder (e.g., "hdbprices.csv")

        Returns:
            polars DataFrame with the loaded data.
        """
        if module not in _MODULES:
            raise ValueError(
                f"Unknown module '{module}'. Available: {sorted(_MODULES)}"
            )

        if self._colab:
            path = self._root / module / filename
            if not path.exists():
                raise FileNotFoundError(
                    f"File not found: {path}. "
                    f"Ensure mlfp_data is accessible in your Google Drive."
                )
        else:
            # Check repo-local data/ first, then fall back to Drive download
            if self._local_data:
                local_path = self._local_data / module / filename
                if local_path.exists():
                    path = local_path
                    logger.info(
                        "Loading %s/%s from local data/ (%s)", module, filename, path
                    )
                    return _read_file(path)
            path = _download_from_drive(module, filename, self._cache)

        logger.info("Loading %s/%s (%s)", module, filename, path)
        return _read_file(path)

    def list_files(self, module: str) -> list[str]:
        """List available data files in a module folder."""
        if module not in _MODULES:
            raise ValueError(
                f"Unknown module '{module}'. Available: {sorted(_MODULES)}"
            )

        if self._colab:
            root = self._root / module
        else:
            root = self._cache / module

        if not root.exists():
            return []

        return sorted(f.name for f in root.iterdir() if f.is_file())

    # -- Module shortcuts --

    @classmethod
    def mlfp01(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp01 (Data Pipelines & Visualisation)."""
        return cls().load("mlfp01", filename)

    @classmethod
    def mlfp02(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp02 (Statistical Mastery)."""
        return cls().load("mlfp02", filename)

    @classmethod
    def mlfp03(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp03 (Supervised ML)."""
        return cls().load("mlfp03", filename)

    @classmethod
    def mlfp04(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp04 (Unsupervised ML)."""
        return cls().load("mlfp04", filename)

    @classmethod
    def mlfp05(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp05 (Deep Learning & Vision)."""
        return cls().load("mlfp05", filename)

    @classmethod
    def mlfp06(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp06 (LLMs, Agents & Transformation)."""
        return cls().load("mlfp06", filename)

    @classmethod
    def assessment(cls, filename: str) -> pl.DataFrame:
        """Load from mlfp_assessment (capstone datasets)."""
        return cls().load("mlfp_assessment", filename)


# ── shared/mlfp04/ex_2.py ──
"""
Shared infrastructure for MLFP04 Exercise 2 — EM and Gaussian Mixture Models.

Contains: synthetic GMM data generation, Singapore e-commerce loader, scaler,
scoring helpers, and OUTPUT_DIR management.

Technique-specific code (the EM E-step / M-step from scratch, BIC/AIC sweeps,
covariance-type comparison, mixture-of-experts gating) does NOT live here.
It lives in the per-technique files under modules/mlfp04/solutions/ex_2/.
"""

from pathlib import Path

import numpy as np
import polars as pl
from sklearn.metrics import silhouette_score
from sklearn.preprocessing import StandardScaler

from kailash_ml.interop import to_sklearn_input


# ════════════════════════════════════════════════════════════════════════
# ENVIRONMENT SETUP
# ════════════════════════════════════════════════════════════════════════

setup_environment()

OUTPUT_DIR = Path("outputs") / "ex2_gmm"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_STATE: int = 42


# ════════════════════════════════════════════════════════════════════════
# SYNTHETIC 2D DATA — 3 well-separated Gaussians
# ════════════════════════════════════════════════════════════════════════
#
# Used to validate the from-scratch EM implementation. Three well-separated
# components make convergence obvious and let students verify the recovered
# parameters against the ground truth.

TRUE_MEANS: np.ndarray = np.array([[0.0, 0.0], [5.0, 2.0], [2.0, 6.0]])
TRUE_COVS: np.ndarray = np.array(
    [
        [[1.0, 0.3], [0.3, 0.8]],
        [[0.8, -0.2], [-0.2, 1.2]],
        [[1.5, 0.0], [0.0, 0.5]],
    ]
)
TRUE_WEIGHTS: np.ndarray = np.array([0.4, 0.35, 0.25])
N_SYNTH: int = 600


def make_synthetic_gmm(
    seed: int = RANDOM_STATE,
) -> tuple[np.ndarray, np.ndarray]:
    """Draw N_SYNTH points from the 3-component GMM defined by TRUE_*.

    Returns (X, z_true) where z_true is the ground-truth component index.
    """
    rng = np.random.default_rng(seed)
    n_per = (TRUE_WEIGHTS * N_SYNTH).astype(int)
    n_per[-1] = N_SYNTH - n_per[:-1].sum()

    parts: list[np.ndarray] = []
    labels: list[int] = []
    for k, (mean, cov, n) in enumerate(zip(TRUE_MEANS, TRUE_COVS, n_per)):
        parts.append(rng.multivariate_normal(mean, cov, n))
        labels.extend([k] * n)

    X = np.vstack(parts)
    z = np.array(labels)

    # Shuffle so the order does not leak the label
    idx = rng.permutation(N_SYNTH)
    return X[idx], z[idx]


# ════════════════════════════════════════════════════════════════════════
# REAL DATA — Singapore e-commerce customers
# ════════════════════════════════════════════════════════════════════════
#
# We reuse the MLFP03 e-commerce customer dataset (~6K rows, Singapore)
# for every real-data task in this exercise. Segmentation is the business
# frame: the GMM will propose soft customer segments that marketing can
# score on expected value.


def load_customers_scaled() -> (
    tuple[np.ndarray, pl.DataFrame, list[str], StandardScaler]
):
    """Return (X_scaled, customers_df, feature_cols, scaler).

    The DataFrame and feature_cols are returned so technique files can
    join cluster labels back onto the original rows for segment profiling.
    """
    loader = MLFPDataLoader()
    customers = loader.load("mlfp03", "ecommerce_customers.parquet")

    numeric_types = (pl.Float64, pl.Float32, pl.Int64, pl.Int32)
    feature_cols = [
        c
        for c, d in zip(customers.columns, customers.dtypes)
        if d in numeric_types and c not in ("customer_id",)
    ]
    customers = customers.drop_nulls(subset=feature_cols)
    X, _, _ = to_sklearn_input(customers, feature_columns=feature_cols)
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X)
    return X_scaled, customers, feature_cols, scaler


# ════════════════════════════════════════════════════════════════════════
# PARAMETER COUNTING — for BIC/AIC interpretation
# ════════════════════════════════════════════════════════════════════════


def count_gmm_params(n_components: int, n_features: int, cov_type: str) -> int:
    """Number of free parameters in a GMM given components, features, cov type.

    Used to explain the BIC/AIC ranking across covariance types.
    """
    d = n_features
    k = n_components
    if cov_type == "full":
        return k * (d * (d + 1) // 2 + d + 1) - 1
    if cov_type == "tied":
        return d * (d + 1) // 2 + k * (d + 1) - 1
    if cov_type == "diag":
        return k * (2 * d + 1) - 1
    if cov_type == "spherical":
        return k * (d + 2) - 1
    raise ValueError(f"Unknown cov_type: {cov_type}")


# ════════════════════════════════════════════════════════════════════════
# SCORING HELPERS
# ════════════════════════════════════════════════════════════════════════


def safe_silhouette(X: np.ndarray, labels: np.ndarray) -> float:
    """Silhouette with a graceful fallback when only one cluster is present."""
    if len(set(labels.tolist())) < 2:
        return float("nan")
    return float(silhouette_score(X, labels))


def out_path(filename: str) -> Path:
    """Return a path under OUTPUT_DIR for a visualisation artefact."""
    return OUTPUT_DIR / filename




# ════════════════════════════════════════════════════════════════════════
# MLFP04 — Exercise 2.4: Mixture of Experts — Soft vs Hard Assignments
# ════════════════════════════════════════════════════════════════════════
#
# WHAT YOU'LL LEARN:
#   - Read GMM soft assignments as intent vectors, not class labels
#   - Measure assignment confidence with max-probability and entropy
#   - Identify boundary customers that hard clustering would bury
#   - Explain Mixture of Experts as input-dependent gating (g_k(x))
#   - Connect classical MoE to Sparse MoE in modern LLMs (Mixtral)
#
# PREREQUISITES: 03_covariance_types.py (BIC-optimal K on customer data)
#
# ESTIMATED TIME: ~35 min
#
# TASKS:
#   1. Theory — MoE as GMM with input-dependent mixing
#   2. Build — soft_vs_hard analysis + simple MoE gate demo
#   3. Train — fit the BIC-optimal GMM and extract soft responsibilities
#   4. Visualise — confidence histogram + segment profile
#   5. Apply — Carousell personalised listing ranking (Singapore)
# ════════════════════════════════════════════════════════════════════════


In [ ]:
from __future__ import annotations

import numpy as np
import polars as pl
from sklearn.mixture import GaussianMixture

from kailash_ml import ModelVisualizer



## THEORY — From GMM to Mixture of Experts


In [ ]:
#
#   GMM:  P(x)   = Sum_k pi_k * N(x | mu_k, Sigma_k)
#                  ^^^^
#                  fixed mixing weights (same for every input)
#
#   MoE:  P(y|x) = Sum_k g_k(x) * f_k(x)
#                  ^^^^^^^       ^^^^^^^
#                  input-dep     expert model k
#                  gating
#
# The gating network g_k(x) is typically a softmax classifier:
#
#     g_k(x) = exp(w_k^T x) / Sum_j exp(w_j^T x)
#
# Instead of saying "40% of all data goes to component 1", MoE says
# "for THIS particular x, 80% of the weight goes to expert 1". The EM
# algorithm generalises: the E-step computes responsibilities given
# the current gate, the M-step fits the experts AND the gate.
#
# MODERN RELEVANCE — Sparse MoE in LLMs:
#   - Mixtral 8x7B has 8 experts of 7B params each, but the router
#     picks only the top-2 per token. Effective compute is ~14B
#     active params, not 56B.
#   - GPT-4 is widely believed to use a Sparse MoE routing scheme
#     for the same reason: decouple model capacity from compute cost.
#   - The gating network in Mixtral is a tiny MLP that reads the
#     token hidden state and outputs 8 routing logits — exactly the
#     g_k(x) from the equation above, just learned end-to-end.



## TASK 2 — BUILD: soft_vs_hard analysis + gate demo


Returns fractions of rows in each confidence band plus mean entropy.


A 2-expert softmax gating demo: route by the sign of the first feature.


In [ ]:
def soft_vs_hard(soft_probs: np.ndarray) -> dict[str, float]:
    max_probs = soft_probs.max(axis=1)
    entropy = -np.sum(soft_probs * np.log(soft_probs + 1e-300), axis=1)
    return {
        "confident": float((max_probs > 0.95).mean()),
        "moderate": float(((max_probs > 0.7) & (max_probs <= 0.95)).mean()),
        "ambiguous": float(((max_probs > 0.5) & (max_probs <= 0.7)).mean()),
        "uncertain": float((max_probs <= 0.5).mean()),
        "mean_entropy": float(entropy.mean()),
        "max_entropy": float(np.log(soft_probs.shape[1])),
    }


def simple_moe_gate(X: np.ndarray, seed: int = 42) -> np.ndarray:
    rng = np.random.default_rng(seed)
    logits = np.column_stack([X[:, 0], -X[:, 0]]) + 0.1 * rng.standard_normal(
        (X.shape[0], 2)
    )
    exp = np.exp(logits - logits.max(axis=1, keepdims=True))
    return exp / exp.sum(axis=1, keepdims=True)



## TASK 3 — TRAIN: fit BIC-optimal GMM and extract soft responsibilities


In [ ]:
X_scaled, customers, feature_cols, _ = load_customers_scaled()

best_k, best_bic = None, float("inf")
for k in range(2, 9):
    gmm = GaussianMixture(n_components=k, covariance_type="full", random_state=42).fit(
        X_scaled
    )
    b = float(gmm.bic(X_scaled))
    if b < best_bic:
        best_bic, best_k = b, k

print("=" * 70)
print(f"  Soft-assignment analysis at BIC-optimal K={best_k}")
print("=" * 70)

best_gmm = GaussianMixture(
    n_components=best_k, covariance_type="full", random_state=42
).fit(X_scaled)
soft_probs = best_gmm.predict_proba(X_scaled)
hard_labels = best_gmm.predict(X_scaled)

profile = soft_vs_hard(soft_probs)
print(f"\nConfidence bands (fraction of customers in each):")
print(f"  confident  (>0.95):     {profile['confident']:.1%}")
print(f"  moderate   (0.70-0.95): {profile['moderate']:.1%}")
print(f"  ambiguous  (0.50-0.70): {profile['ambiguous']:.1%}")
print(f"  uncertain  (<=0.50):    {profile['uncertain']:.1%}")
print(
    f"\nMean entropy: {profile['mean_entropy']:.4f}  "
    f"(max possible: {profile['max_entropy']:.4f})"
)



### Checkpoint 1


In [ ]:
assert soft_probs.shape == (X_scaled.shape[0], best_k), "R must be (n, K)"
assert abs(soft_probs.sum(axis=1).mean() - 1.0) < 1e-4, "rows must sum to 1"
assert soft_probs.min() >= 0, "responsibilities non-negative"
print("\n[ok] Checkpoint 1 passed — responsibilities form a valid distribution")

# Boundary customers — the ones hard clustering would miss
boundary_mask = soft_probs.max(axis=1) < 0.6
n_boundary = int(boundary_mask.sum())
print(
    f"\nBoundary customers (max responsibility < 0.6): "
    f"{n_boundary} / {X_scaled.shape[0]} "
    f"({n_boundary / X_scaled.shape[0]:.1%})"
)
print(
    "  These rows sit between two or more segments. Hard clustering "
    "would force them into one bucket and bury the cross-segment signal."
)



## TASK 4 — VISUALISE: confidence bands + segment profile


In [ ]:
viz = ModelVisualizer()
confidence_chart = {
    "Confidence bands": {
        "confident": profile["confident"],
        "moderate": profile["moderate"],
        "ambiguous": profile["ambiguous"],
        "uncertain": profile["uncertain"],
    }
}
fig = viz.metric_comparison(confidence_chart)
fig.update_layout(title="Soft-assignment confidence distribution")
fig.write_html(str(out_path("ex2_soft_confidence.html")))
print(f"\nSaved: {out_path('ex2_soft_confidence.html')}")

# Join labels back onto the customer table for per-segment profiling
customers_with_seg = customers.with_columns(
    pl.Series("gmm_segment", hard_labels),
    pl.Series("gmm_confidence", soft_probs.max(axis=1)),
)

print("\nPer-segment summary (hard-label counts + soft mass):")
for k in range(best_k):
    subset = customers_with_seg.filter(pl.col("gmm_segment") == k)
    hard_count = subset.height
    soft_mass = float(soft_probs[:, k].sum())
    print(
        f"  Segment {k}: hard={hard_count:>5}  "
        f"soft_mass={soft_mass:>8.1f}  "
        f"weight={best_gmm.weights_[k]:.3f}"
    )



### Checkpoint 2


In [ ]:
assert customers_with_seg.height == X_scaled.shape[0], "join preserved all rows"
assert out_path("ex2_soft_confidence.html").exists(), "confidence chart written"
print("\n[ok] Checkpoint 2 passed — per-segment profile produced")

# MoE gate demo on synthetic 2D data
moe_demo = simple_moe_gate(X_scaled[:, :2])
print(f"\nMoE gating demo on first 2 customer features:")
print(f"  Expert 0 active (gate > 0.5): {(moe_demo[:, 0] > 0.5).mean():.1%}")
print(f"  Expert 1 active (gate > 0.5): {(moe_demo[:, 1] > 0.5).mean():.1%}")



### Checkpoint 3


In [ ]:
assert moe_demo.shape == (X_scaled.shape[0], 2), "MoE gate must produce 2 weights"
assert abs(moe_demo.sum(axis=1).mean() - 1.0) < 1e-4, "gate must be a distribution"
print("[ok] Checkpoint 3 passed — MoE gate produces a valid softmax distribution")



## TASK 5 — APPLY: Carousell Personalised Listing Ranking (Singapore)


In [ ]:
# SCENARIO: Carousell (Singapore) is the region's largest C2C
# marketplace with ~35M users. When a shopper opens the app, the feed
# ranker has ~80ms to produce a personalised ordering from ~10M live
# listings. Pure classification is too slow at that scale — and the
# shopper's intent is rarely one-dimensional.
#
# Why MoE-style routing beats hard segmentation:
#   - A shopper on a Saturday morning may be 60% "bargain hunter", 30%
#     "home-decor browser", 10% "gift shopper" in the same session.
#   - A hard segment model would pick one label and surface only that
#     vertical's listings — the shopper bounces back to search.
#   - A soft-responsibility ranker computes expected-click-through as
#     sum_k r_k * CTR_k(listing), which blends the three intents and
#     ranks items that ANY of the three would engage with at the top.
#
# The MoE pattern generalises: replace the k Gaussians with k ranking
# "experts" (each specialised for an intent vertical) and the gating
# network with a tiny MLP over session features. That is exactly what
# YouTube, Pinterest, and Carousell's own production rankers use.
#
# BUSINESS IMPACT:
#   - Feed impressions/day: ~900M
#   - Baseline click-through rate: ~6.2% (Carousell public disclosures)
#   - Internal A/B tests on Southeast Asian marketplaces have lifted
#     CTR by ~8% when switching from hard-segment ranking to soft
#     intent-vector ranking at the same serving cost.
#   - 8% CTR lift on 900M impressions/day = ~4.5M extra clicks/day.
#     At Carousell's ~S$0.018 average monetisation per click, that is
#     ~S$81,000/day = S$29.6M/year in additional take-rate revenue.
#   - Zero marginal infra cost — the GMM is fitted offline nightly
#     and the soft responsibilities are materialised into the same
#     feature store the existing ranker already reads.

print("\n" + "=" * 70)
print("  APPLY — Carousell personalised listing ranking")
print("=" * 70)
print(
    f"Out of {X_scaled.shape[0]} customers, {n_boundary} ({n_boundary / X_scaled.shape[0]:.1%}) "
    "have mixed intent. Soft scoring keeps ALL of them in the long-tail "
    "of every applicable segment ranker, instead of burying them in one."
)
print(
    "At Carousell's scale, blending intents with soft responsibilities "
    "recovers ~S$29.6M/year in feed monetisation — from the same GMM "
    "you just fitted, read in a different way."
)



## REFLECTION


[x] Soft GMM responsibilities carry uncertainty hard labels destroy
  [x] Max-probability and entropy diagnose boundary customers
  [x] MoE = GMM with input-dependent gating g_k(x)
  [x] Sparse MoE in Mixtral/GPT-4 is the same idea at LLM scale
  [x] Carousell scenario: soft intent vectors unlock S$29.6M/year
      in feed revenue without extra serving cost

  KEY INSIGHT: the GMM you just fitted is already a personalisation
  engine. You don't need a new model — you need a new way to READ the
  responsibility matrix. Hard argmax throws away 80% of the signal.

  Exercise 2 complete. Next: Exercise 3 introduces PCA and
  dimensionality reduction on the same customer data.


In [ ]:
print("\n" + "=" * 70)
print("  WHAT YOU'VE MASTERED")
print("=" * 70)
print(
)

